<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="assets/content/images/titanic_thumbnail.png" align="center" width="20%">
</div>

<br>

# TITANIC: EXPLORATORY DATA ANALYSIS AND FEATURE ENGINEERING

<br>

**About:** A structured walkthrough of exploratory data analysis and feature engineering on the Titanic passenger dataset - from raw tabular data to a clean, model-ready feature matrix.

**Learning Goals:** After completing this notebook, you will be able to:
1. Explore a real dataset using distribution plots, survival-rate comparisons, and summary statistics
2. Identify and handle missing values using group-based median imputation
3. Extract new categorical features from free-text columns using regular expressions
4. Encode categorical variables into binary and ordinal representations
5. Evaluate feature relevance through a Pearson correlation heatmap

**Keywords:** exploratory data analysis, feature engineering, missing value imputation, one-hot encoding, Titanic

**Prerequisite Knowledge:** (1) Python and pandas basics (DataFrames, groupby, loc/iloc), (2) Descriptive statistics (mean, median, distributions)

**Target User:** Data science learners who know pandas and want hands-on practice with real-world feature engineering on a classic binary classification problem

<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 1: DATASET OVERVIEW AND LOADING](#Part_1)
> #### [PART 2: EXPLORATORY DATA ANALYSIS](#Part_2)
> #### [PART 3: FEATURE ENGINEERING](#Part_3)
> #### [PART 4: FINAL FEATURE SET AND CORRELATION ANALYSIS](#Part_4)

<br>

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **DATASET** OVERVIEW AND LOADING

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/titanic_ship.png" align="center" width="40%" padding="10"><br>
    <br>
</div>

#### CONTENTS:

> [PART 1.1: The Titanic Dataset](#Part_1_1)<br>
> [PART 1.2: Loading and Inspecting the Data](#Part_1_2)<br>

<a id='Part_1_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.1: THE TITANIC DATASET

<br>

## **Understanding** the Data

The sinking of the RMS Titanic on April 15, 1912 claimed 1,502 of 2,224 lives. Survival was not random: access to lifeboats was rationed by class, crew protocols directed women and children first, and the ship's layout gave first-class passengers faster routes to the deck.

This dataset - originally published as the introductory Kaggle competition - contains records for 891 passengers in the training set and 418 in the test set. Each record lists passenger attributes; the training set includes a `Survived` column (1 = survived, 0 = did not survive) that we use as our prediction target.

Our goal in this notebook is to transform the raw attributes into a clean, numeric feature matrix. A companion notebook (`02_ensemble_methods_and_automl.ipynb`) then uses AutoML frameworks to predict survival and combine their predictions into a simple ensemble.

___

**Note:** The Titanic dataset is a teaching benchmark. The patterns here reflect one historical incident with real confounds - for example, first-class tickets included better cabin locations that reduced evacuation time, so `Pclass` captures both socioeconomic and physical factors simultaneously.

___

**Sources Consulted:**
- Titanic dataset and variable descriptions: [Kaggle Titanic Competition](https://www.kaggle.com/c/titanic/data)

<a id='Part_1_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.2: LOADING AND INSPECTING THE DATA

<br>

**Imports and Configuration**

All imports are declared here so any reader can see the full dependency surface at a glance. The `plot_distribution` helper below creates a faceted KDE plot that we use to compare feature distributions between survivors and non-survivors.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

# Plot styling
sns.set(style='white', context='notebook', palette='deep')
plt.rcParams['figure.figsize'] = 10, 6
pd.set_option('display.max_columns', 100)

def plot_distribution(df, var, target, **kwargs):
    row = kwargs.get('row', None)
    col = kwargs.get('col', None)
    facet = sns.FacetGrid(df, hue=target, aspect=4, row=row, col=col)
    facet.map(sns.kdeplot, var, fill=True)
    facet.set(xlim=(0, df[var].max()))
    facet.add_legend()
    plt.tight_layout()

**Loading the Data**

We load both splits upfront and keep them in sync through a `combine` list. Transformations applied to one dataset (e.g., encoding a column) need to be applied to the other as well - mutating both at once via `combine` prevents the two from drifting apart.

In [ ]:
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')
combine  = [train_df, test_df]   # any transform that touches columns applies to both

print(f"Training set: {train_df.shape[0]} rows, {train_df.shape[1]} columns")
print(f"Test set:     {test_df.shape[0]} rows, {test_df.shape[1]} columns")

**Initial Inspection**

The first pass answers three questions: What columns exist and what are their types? Are there any missing values? What do typical values look like?

In [ ]:
# Column names and data types
print("Columns and dtypes:")
print(train_df.dtypes)
print()

# First five rows
train_df.head(5)

In [ ]:
# Summary statistics for numeric columns
train_df.describe()

In [ ]:
# Missing value counts for each column
print("Training set - missing values:")
print(train_df.isnull().sum())
print()
print("Test set - missing values:")
print(test_df.isnull().sum())

**Observations:**

- `Age` is missing for about 20% of training rows and 21% of test rows - too many to drop, so we will impute.
- `Cabin` is missing for 77% of training rows - too sparse to use directly; we will either extract a coarse cabin-deck indicator or drop it.
- `Embarked` is missing for only 2 training rows - easy to fill with the mode.
- `Fare` is missing for 1 test row - fill with the median.
- All other columns are complete.

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **Load `test.csv` and calculate what percentage of test passengers are missing an `Age` value. Then, without looking ahead: propose one alternative imputation strategy to the group-median approach used in Part 3, and name one situation where your alternative would be better.**

<br>

In [ ]:
# Your code here
# Hint: use test_df['Age'].isnull().sum() and len(test_df)


<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **EXPLORATORY** DATA ANALYSIS

<div align="center" style="font-size:12px; font-family:FreeMono; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/eda_distributions.png" align="center" width="40%" padding="10"><br>
    <br>
</div>

#### CONTENTS:

> [PART 2.1: Feature Distributions](#Part_2_1)<br>
> [PART 2.2: Target Balance and Baseline Accuracy](#Part_2_2)<br>
> [PART 2.3: Removing Redundant Features](#Part_2_3)<br>

<a id='Part_2_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.1: FEATURE DISTRIBUTIONS

<br>

Histograms give a fast overview of value ranges and skewness for every numeric column at once. We are looking for: columns with unusual scales that might need normalization, bimodal distributions that might suggest hidden subgroups, and outliers.

In [ ]:
train_df.hist(figsize=(13, 10))
plt.suptitle("Training Set - Feature Distributions", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

**Observations:**

- `Age` is approximately normally distributed with a right tail - a few elderly passengers.
- `Fare` is strongly right-skewed - a small number of first-class passengers paid very high fares. We will bin it into quartiles later.
- `Pclass` has three distinct values (1, 2, 3) - already ordinal, but survival rate differs substantially across classes.
- `SibSp` and `Parch` are sparse counts; most passengers traveled alone or with small families.

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.2: TARGET BALANCE AND BASELINE ACCURACY

<br>

Before modeling, it is important to understand the target class distribution. An imbalanced target shifts how we interpret accuracy: a model that always predicts "did not survive" would be ~62% accurate on this data without learning anything.

In [ ]:
target_count = train_df['Survived'].value_counts()
print("Survival counts:")
print(target_count)
print()

# Majority-class baseline accuracy
baseline = target_count[0] / target_count.sum()
print(f"Majority-class baseline accuracy: {baseline:.1%}")
print("Any model that scores below this is worse than a trivial classifier.")

In [ ]:
# Survival rate by passenger class
print("Survival rate by Pclass:")
print(train_df.groupby('Pclass')['Survived'].mean().rename('survival_rate').round(3))
print()

# Survival rate by sex
print("Survival rate by Sex:")
print(train_df.groupby('Sex')['Survived'].mean().rename('survival_rate').round(3))

___

**Note:** The ~38% survival rate reflects the documented "women and children first" evacuation protocol and the limited number of lifeboats (capacity for roughly 1,178 people against 2,224 aboard). `Sex` and `Pclass` are among the strongest individual predictors for exactly these structural reasons.

___

**Sources Consulted:**
- Titanic lifeboat statistics: [Encyclopedia Titanica - Lifeboats](https://www.encyclopedia-titanica.org/titanic-lifeboats.html)

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.3: REMOVING REDUNDANT FEATURES

<br>

`PassengerId` is an arbitrary index assigned by the dataset publisher - it carries no information about the passenger. Including it would let a model overfit to row ordering rather than passenger attributes. We drop it from training data only; the test set retains it so we can identify predictions during submission.

In [ ]:
print("Shapes before drop:", train_df.shape, test_df.shape)

train_df = train_df.drop(['PassengerId'], axis=1)
combine  = [train_df, test_df]

print("Shapes after drop: ", train_df.shape, test_df.shape)

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **Calculate the survival rate separately for first-class passengers and third-class passengers. Then calculate the survival rate for female passengers and male passengers. Which single feature - `Pclass` or `Sex` - shows a larger absolute difference in survival rate between its groups?**

<br>

In [ ]:
# Your code here


<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **FEATURE** ENGINEERING

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/feature_engineering.png" align="center" width="40%" padding="10"><br>
    <br>
</div>

Feature engineering transforms raw attributes into representations that reveal patterns a model can exploit. This Part covers eight techniques applied to the Titanic data: regex extraction, categorical encoding, group-based imputation, discretization, interaction terms, mode-fill, missing-value indicators, and feature pruning.

#### CONTENTS:

> [PART 3.1: Title Extraction from Name](#Part_3_1)<br>
> [PART 3.2: Sex Encoding](#Part_3_2)<br>
> [PART 3.3: Age Imputation and Discretization](#Part_3_3)<br>
> [PART 3.4: Travel Party Size](#Part_3_4)<br>
> [PART 3.5: Interaction Feature (Age x Class)](#Part_3_5)<br>
> [PART 3.6: Port of Embarkation](#Part_3_6)<br>
> [PART 3.7: Fare Binning](#Part_3_7)<br>
> [PART 3.8: Cabin and Ticket](#Part_3_8)<br>

<a id='Part_3_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.1: TITLE EXTRACTION FROM NAME

<br>

The `Name` column contains the passenger's full name including a title (Mr., Mrs., Miss., Master., etc.). Titles encode social status and age category simultaneously: "Master" was used for boys under ~14, "Miss" for unmarried women, "Mrs." for married women. These distinctions correlate with survival because evacuation priority was given partly by social category, not raw age.

We use a regular expression to extract the title: match a space, one or more letters, and a period.

In [ ]:
# Preview names to verify the pattern
train_df['Name'].head(5)

In [ ]:
# Extract title using regex: space + letters + period
for dataset in combine:
    dataset['Title'] = dataset['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# Cross-check titles against Sex to catch encoding errors
pd.crosstab(train_df['Title'], train_df['Sex']).head(10)

Most passengers have one of four titles: Mr., Mrs., Miss., Master. The remainder ("Lady", "Countess", "Dr.", "Rev.", etc.) are rare enough that grouping them avoids overfitting to title-specific patterns that may not generalize.

In [ ]:
rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr',
               'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
for dataset in combine:
    dataset['Title'] = dataset['Title'].replace(rare_titles, 'Rare')
    dataset['Title'] = dataset['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

# Survival rate by title confirms titles carry predictive signal
train_df[['Title', 'Survived']].groupby('Title').mean().sort_values('Survived', ascending=False)

In [ ]:
# Plot survival count by title
sns.countplot(x='Survived', hue='Title', data=train_df, order=[1, 0])
plt.xticks([0, 1], ['Survived', 'Did Not Survive'])
plt.title("Survival Count by Title")
plt.show()

In [ ]:
# One-hot encode Title into binary columns, then drop original Name and Title
for dataset in combine:
    dummies = pd.get_dummies(dataset['Title'])
    dataset[dummies.columns] = dummies

train_df = train_df.drop(['Name', 'Title'], axis=1)
test_df  = test_df.drop(['Name', 'Title'], axis=1)
combine  = [train_df, test_df]

train_df.head(3)

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.2: SEX ENCODING

<br>

Most models require numeric inputs. Binary encoding (female = 1, male = 0) is sufficient here because `Sex` has exactly two categories with no natural ordering between them. One-hot encoding would produce two columns that are perfectly anti-correlated - adding no information.

In [ ]:
for dataset in combine:
    dataset['Sex'] = dataset['Sex'].map({'female': 1, 'male': 0}).astype(int)

print("Survival rate by Sex (0=male, 1=female):")
print(train_df.groupby('Sex')['Survived'].mean().rename('survival_rate').round(3))
train_df.head(3)

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.3: AGE IMPUTATION AND DISCRETIZATION

<br>

Simple mean or median imputation for `Age` ignores the fact that age varies by sex and class: a first-class female and a third-class male are unlikely to have the same age distribution. We estimate the median age for each of the 6 (sex x class) subgroups and fill missing values with the appropriate subgroup median.

This approach is more principled than a global median because it preserves the conditional distribution of age - a better approximation of what the missing value probably was.

___

**Note:** The row index of `guess_ages` is sex (0=male, 1=female) and the column index is Pclass - 1 (0=first, 1=second, 2=third).

___

In [ ]:
guess_ages = np.zeros((2, 3), dtype=int)

for idx, dataset in enumerate(combine):
    label = "Training" if idx == 0 else "Test"
    print(f"--- {label} set ---")
    for i in range(2):       # sex: 0=male, 1=female
        for j in range(3):   # pclass: 1, 2, 3 -> index 0, 1, 2
            subset = dataset[(dataset['Sex'] == i) & (dataset['Pclass'] == j + 1)]['Age'].dropna()
            guess_ages[i, j] = int(subset.median())

    print("Median age by [sex, pclass-1]:\n", guess_ages)

    # Fill NaN Age values using the subgroup median
    for i in range(2):
        for j in range(3):
            dataset.loc[(dataset['Age'].isnull()) &
                        (dataset['Sex'] == i) &
                        (dataset['Pclass'] == j + 1), 'Age'] = guess_ages[i, j]

    dataset['Age'] = dataset['Age'].astype(int)
    print()

print("Age NaN remaining in train:", train_df['Age'].isnull().sum())

After imputation we discretize `Age` into 5 bands. Discretization reduces sensitivity to the exact imputed value (imputing 22 vs 24 matters less once both map to the same band) and lets the model learn non-linear age effects without polynomial features.

In [ ]:
# Check survival rate by AgeBand before mapping to integers
train_df['AgeBand'] = pd.cut(train_df['Age'], 5)
print(train_df[['AgeBand', 'Survived']].groupby('AgeBand', observed=True)
      .mean().sort_values('AgeBand'))

# Distribution of Age vs Survived split by Sex (Sex: 0=male, 1=female)
plot_distribution(train_df, var='Age', target='Survived', row='Sex')

In [ ]:
# Map continuous age to ordinal band (0-4)
for dataset in combine:
    dataset.loc[dataset['Age'] <= 16, 'Age'] = 0
    dataset.loc[(dataset['Age'] > 16) & (dataset['Age'] <= 32), 'Age'] = 1
    dataset.loc[(dataset['Age'] > 32) & (dataset['Age'] <= 48), 'Age'] = 2
    dataset.loc[(dataset['Age'] > 48) & (dataset['Age'] <= 64), 'Age'] = 3
    dataset.loc[dataset['Age'] > 64, 'Age'] = 4
    dataset['Age'] = dataset['Age'].astype(int)

train_df = train_df.drop(['AgeBand'], axis=1)
combine  = [train_df, test_df]
train_df[['Age', 'Survived']].groupby('Age').mean().rename(columns={'Survived': 'survival_rate'})

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3_4'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.4: TRAVEL PARTY SIZE

<br>

`SibSp` (siblings/spouses aboard) and `Parch` (parents/children aboard) individually describe family relationships. Their sum plus 1 (the passenger themselves) gives total travel party size. The key hypothesis: very small parties (size 1 = traveling alone) and very large parties may have had different survival dynamics than medium-sized families.

In [ ]:
for dataset in combine:
    dataset['FamilySize'] = dataset['SibSp'] + dataset['Parch'] + 1

print("Survival rate by FamilySize:")
print(train_df[['FamilySize', 'Survived']].groupby('FamilySize')
      .mean().rename(columns={'Survived': 'survival_rate'}).round(3))

sns.countplot(x='Survived', hue='FamilySize', data=train_df, order=[1, 0])
plt.xticks([0, 1], ['Survived', 'Did Not Survive'])
plt.title("Survival Count by Family Size")
plt.show()

The data supports that solo travelers and very large families had lower survival rates, while families of 2-4 had the highest. We collapse this into a binary `IsAlone` indicator, which captures most of the signal without introducing a high-cardinality ordinal feature.

In [ ]:
for dataset in combine:
    dataset['IsAlone'] = (dataset['FamilySize'] == 1).astype(int)
    dataset.drop(['Parch', 'SibSp', 'FamilySize'], axis=1, inplace=True)

print("Survival rate by IsAlone (0=with family, 1=alone):")
print(train_df[['IsAlone', 'Survived']].groupby('IsAlone')
      .mean().rename(columns={'Survived': 'survival_rate'}).round(3))
train_df.head(3)

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3_5'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.5: INTERACTION FEATURE (AGE x CLASS)

<br>

Interaction features encode the joint effect of two variables when that joint effect is stronger than either variable alone. A young child in third class and a young child in first class had very different survival chances - the `Age*Class` product captures this because it will be small for young first-class passengers and larger for young third-class passengers.

In [ ]:
for dataset in combine:
    dataset['Age*Class'] = dataset['Age'] * dataset['Pclass']

print("Survival rate by Age*Class interaction:")
print(train_df[['Age*Class', 'Survived']].groupby('Age*Class')
      .mean().rename(columns={'Survived': 'survival_rate'}).round(3))

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3_6'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.6: PORT OF EMBARKATION

<br>

Passengers boarded at three ports: Southampton (S), Cherbourg (C), and Queenstown (Q). Port correlates with passenger class (Cherbourg had proportionally more first-class passengers) so it may carry indirect survival signal.

Two rows in the training set are missing `Embarked`. We fill with the mode - the most frequent port - rather than imputing based on other features, because the missingness is so sparse that any more complex approach is not justified.

In [ ]:
freq_port = train_df['Embarked'].dropna().mode()[0]
print(f"Most frequent port: {freq_port}")

for dataset in combine:
    dataset['Embarked'] = dataset['Embarked'].fillna(freq_port)

print("Survival rate by Embarked:")
print(train_df[['Embarked', 'Survived']].groupby('Embarked')
      .mean().sort_values('Survived', ascending=False).round(3))

In [ ]:
sns.countplot(x='Survived', hue='Embarked', data=train_df, order=[1, 0])
plt.xticks([0, 1], ['Survived', 'Did Not Survive'])
plt.title("Survival Count by Port of Embarkation")
plt.show()

In [ ]:
# One-hot encode Embarked, then drop the original column
for dataset in combine:
    dummies = pd.get_dummies(dataset['Embarked'])
    dataset[dummies.columns] = dummies
    dataset.drop('Embarked', axis=1, inplace=True)

train_df.head(3)

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3_7'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.7: FARE BINNING

<br>

`Fare` is right-skewed - one outlier ticket paid over £500 while most paid under £30. Feeding raw fares to a linear model amplifies the outlier's influence. We use quantile binning (`pd.qcut`) to split fares into 4 equal-frequency bins, converting fare into an ordinal 0-3 rank. This makes the feature robust to the skew while preserving the rank ordering.

In [ ]:
# One test row is missing Fare - fill with training median
train_df['Fare'].fillna(train_df['Fare'].median(), inplace=True)
test_df['Fare'].fillna(train_df['Fare'].median(), inplace=True)

# Preview survival by FareBand
train_df['FareBand'] = pd.qcut(train_df['Fare'], 4)
print(train_df[['FareBand', 'Survived']].groupby('FareBand', observed=True)
      .mean().rename(columns={'Survived': 'survival_rate'}).round(3))

In [ ]:
# Replace continuous Fare with ordinal bin index
for dataset in combine:
    dataset['Fare'] = pd.qcut(dataset['Fare'], 4, labels=np.arange(4)).astype(int)

train_df = train_df.drop(['FareBand'], axis=1)
combine  = [train_df, test_df]
train_df[['Fare', 'Survived']].groupby('Fare').mean().rename(columns={'Survived': 'survival_rate'})

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3_8'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.8: CABIN AND TICKET

<br>

**Cabin**

Cabin is missing for 77% of training passengers. The non-null values encode the deck letter (e.g., "C85" is deck C). Even this partial information shows survival differences by deck - but 77% missingness means any deck indicator is informative only for the minority of passengers who have it, and the majority would receive the same imputed code regardless. We drop Cabin to avoid introducing a feature that is effectively a noise flag for most rows.

**Ticket**

Ticket identifiers are unique strings with no consistent format across passengers. They do not encode any attribute we can extract reliably, and using them directly would cause the model to memorize ticket IDs rather than learn generalizable patterns. We drop Ticket as well.

In [ ]:
# Cabin: examine the deck letter distribution for non-null rows
train_df['CabinDeck'] = train_df['Cabin'].apply(lambda x: str(x)[0] if pd.notnull(x) else 'Unknown')
print("Cabin deck distribution (including Unknown for missing):")
print(train_df['CabinDeck'].value_counts())

sns.countplot(x='Survived', hue='CabinDeck', data=train_df.sort_values('CabinDeck'), order=[1, 0])
plt.title("Survival Count by Cabin Deck (Unknown = missing)")
plt.show()

In [ ]:
# Drop Cabin and CabinDeck from both datasets; drop Ticket as well
train_df = train_df.drop(['Cabin', 'CabinDeck', 'Ticket'], axis=1)
test_df  = test_df.drop(['Cabin', 'Ticket'], axis=1)
combine  = [train_df, test_df]

print("Final training columns:", list(train_df.columns))

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **The notebook creates a binary `IsAlone` feature and discards `FamilySize`. Rewrite the code to instead create a 3-category `PartySize` feature: 0 for alone, 1 for small party (2-4 members), and 2 for large party (5+). Print the survival rate for each category. Does the 3-category version reveal a pattern that the binary version hides?**

<br>

In [ ]:
# Reload fresh columns to demonstrate - create a temporary copy
temp = pd.read_csv('train.csv').copy()
temp['FamilySize'] = temp['SibSp'] + temp['Parch'] + 1

# Your code here: create PartySize (0, 1, 2) and show survival rate per category


<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **FINAL FEATURE** SET AND CORRELATION ANALYSIS

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/correlation_heatmap.png" align="center" width="40%" padding="10"><br>
    <br>
</div>

## **Preprocessing Complete**

All features are now numeric, missing values have been filled, and high-cardinality string columns have been dropped or encoded. The feature engineering steps we applied are summarized below:

| Original Column | Action | Resulting Column(s) |
|---|---|---|
| `Name` | Regex extract title, one-hot encode, then drop | `Master`, `Miss`, `Mr`, `Mrs`, `Rare` |
| `Sex` | Binary encode | `Sex` (0/1) |
| `Age` | Group-median imputation, then 5-band discretization | `Age` (0-4) |
| `SibSp`, `Parch` | Combine into FamilySize, then binarize | `IsAlone` |
| `Ticket` | Drop (no extractable structure) | - |
| `Fare` | Quantile bin into 4 ranks | `Fare` (0-3) |
| `Cabin` | Drop (77% missing) | - |
| `Embarked` | Mode-fill 2 missing rows, one-hot encode | `C`, `Q`, `S` |
| `Pclass`, `Age` | Multiply | `Age*Class` |
| `PassengerId` | Drop (index only) | - |

In [ ]:
# Final feature matrix
print("Training shape:", train_df.shape)
print("Test shape:    ", test_df.shape)
print()
print("Training columns:", list(train_df.columns))
train_df.head(7)

In [ ]:
test_df.head(7)

**Correlation Heatmap**

Pearson correlation measures linear relationships between pairs of features on a -1 to +1 scale. For a classification problem with a binary target, the correlation between each feature and `Survived` tells us how much linear signal that feature carries. High inter-feature correlations (multicollinearity) can destabilize linear models but matter less for tree-based methods.

In [ ]:
colormap = plt.cm.viridis
plt.figure(figsize=(14, 14))
plt.title('Pearson Correlation of Features', y=1.02, size=15)
sns.heatmap(
    train_df.corr().round(2),
    linewidths=0.1, vmax=1.0, vmin=-1.0,
    square=True, cmap=colormap, linecolor='white', annot=True
)
plt.tight_layout()
plt.show()

**Reading the heatmap:**

- `Sex` and `Mrs` are positively correlated with `Survived` - confirming the "women and children first" pattern.
- `Mr` is negatively correlated with `Survived` - adult males had significantly lower survival rates.
- `Pclass` and `Fare` are negatively correlated with each other (lower class number = higher fare), which is expected.
- `Age*Class` shows moderate positive correlation with `Mr` and negative correlation with `Survived` - young passengers in low classes had the worst outcomes.

In the companion notebook (`02_ensemble_methods_and_automl.ipynb`), we feed this feature matrix directly to AutoML frameworks and build a simple ensemble.

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **Look at the correlation heatmap. Name two feature pairs (other than `Pclass`/`Fare`) that have an absolute Pearson correlation above 0.3, and for each pair explain in one sentence why that correlation makes intuitive sense given what you know about the Titanic dataset.**

<br>

In [ ]:
# Compute correlation matrix and filter pairs with |r| > 0.3
corr = train_df.corr().abs()

# Your code here: find and print feature pairs where |r| > 0.3 (excluding self-correlations)


<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<hr style="border: 6px solid#003262;" />